# Chuẩn bị cache quan hệ VSR v2 — CPU
**Accelerator: None · Internet ON · Save & Run All. Không GPU, không train.**

Chỉ sao chép feature CLIP và embedding prompt **gốc** đã có cho612ảnh.
Không tạo detector, CLIP hoặc captioner; không đổi nhãn VSR. Audit nhãn vẫn chờ người xác nhận.

Add Input ba dataset: `visual-cache`, `prompt-cache`, `objectdetectionecache`.
Sửa ba đường dẫn ở cell cấu hình nếu Kaggle mount khác; không đổi nguồn prompt sang UPT/VSR.
Dùng khoảng1GiB output trống; RAM cần đủ để mở cache prompt đầy đủ, tương tự preparation trước.
Không cần checkpointA, ảnhRGB, API token hoặc dataset VSR riêng cho bước này.


In [ ]:
from pathlib import Path
VISUAL=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
LEGACY=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/relation_frames_v2_cache')
REVIEW=Path('/kaggle/working/relation_frames_v2_cache_review.zip')


In [ ]:
import hashlib,json,os,subprocess,sys,tempfile,urllib.request,zipfile
os.environ['CUDA_VISIBLE_DEVICES']=''
SOURCE_COMMIT='dd49cbb9bf281b69b733bb06a818cffbbc24164e'
ASSETS={'prepare_frames_cpu.py': {'path': 'experiments/relation_frames_inventory/prepare_frames_cpu.py', 'sha256': '2a525383ea9d1c791a276fd55138b70e533602db7c866cad193ad03e5f17fa30'}, 'visual_cache.py': {'path': 'captioning/visual_cache.py', 'sha256': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1'}, 'semantic_prompts.py': {'path': 'captioning/semantic_prompts.py', 'sha256': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511'}, 'relation_frames_manifest.bin': {'path': 'experiments/relation_frames_inventory/V2_MANIFEST.json.gz', 'sha256': '6ca6af23d0a31043fecdf33398f5b60098be07cbeea9fb91cbfec4b314c2cfa7'}}
SOURCE=Path(tempfile.mkdtemp(prefix='relation-frames-v2-cpu-'))
LOG=Path('/kaggle/working/relation_frames_v2_cache.log')
if not VISUAL.is_dir():raise FileNotFoundError(VISUAL)
for p in (LEGACY,DETECTIONS):
    if not p.is_file():raise FileNotFoundError(p)
if OUTPUT.exists():raise FileExistsError('Giữ output cũ; đổi sang runtime/output mới trước khi chạy lại.')
for name,entry in ASSETS.items():
    url='https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+SOURCE_COMMIT+'/'+entry['path']
    print('Fetching pinned source:',name,flush=True)
    raw=urllib.request.urlopen(url,timeout=60).read()
    if hashlib.sha256(raw).hexdigest()!=entry['sha256']:raise ValueError('Source hash differs: '+name)
    (SOURCE/name).write_bytes(raw)
print('Pinned source verified. This job copies caches on CPU; no model forward.',flush=True)


In [ ]:
args=[sys.executable,'-u',str(SOURCE/'prepare_frames_cpu.py'),
      '--manifest',str(SOURCE/'relation_frames_manifest.bin'),'--visual',str(VISUAL),
      '--legacy',str(LEGACY),'--detections',str(DETECTIONS),'--output',str(OUTPUT)]
returncode=None
try:
    with LOG.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        returncode=process.wait()
finally:
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        if LOG.is_file():z.write(LOG,'notebook.log')
        if OUTPUT.is_dir():
            for p in OUTPUT.iterdir():
                if p.is_file() and p.suffix in ('.json','.bin'):z.write(p,p.name)
        z.writestr('execution_status.json',json.dumps(dict(returncode=returncode,source_commit=SOURCE_COMMIT,
            neural_forward=False,caption_training=False,training_approved=False),indent=2))
    print('SEND SMALL REVIEW ZIP:',REVIEW,flush=True)
if returncode!=0:raise RuntimeError('CPU cache preparation failed. Send review ZIP/log; do not substitute prompts or relabel data.')
receipt=json.loads((OUTPUT/'receipt.json').read_text())
if receipt['status']!='CACHE_COPY_COMPLETE_PENDING_LABEL_AUDIT':raise RuntimeError('Missing completion receipt')
print('KEEP ENTIRE FOLDER:',OUTPUT,flush=True)
print('Prepared original caches only. Label audit and model/training preflight remain pending.',flush=True)


Giữ toàn bộ `relation_frames_v2_cache` để dùng lại; gửi `relation_frames_v2_cache_review.zip`.
`relation_frames_manifest.bin` là metadata gzip giữ nguyên byte; dùng đuôi.bin để tránh Kaggle tự giải nén.
Thành công của notebook chỉ xác nhận cache đã sao chép, chưa xác nhận chất lượng nhãn hoặc mô hình.
